# Phase 3 - Mini-3D-Recon Training

Trains `MiniReconModel` (shared MobileNetV3-Small backbone + depth/pose heads,
`src/reconstruction/model.py`) on UnityCam's real depth+pose GT only (see
PROGRESS.md for why: matches README's literal phase-table wording, and
sidesteps the unresolved real-cam/UnityCam pose coordinate-frame mismatch).
Pose is parameterized as consecutive frame-to-frame relative SE(3) transforms.

Model/loss/train.py were written against facts confirmed in
`phase3a_pose_explore` (pose format) and `phase3b_depth_explore` (depth
format), then validated locally with real pretrained weights and a fake
in-memory dataset -- see PROGRESS.md.

**GPU fix**: same P100/no-Pascal-kernels issue as Phase 2 -- Kaggle's
API-pushed kernels default to a Tesla P100 (compute capability 6.0), and
Kaggle's preinstalled torch has zero Pascal (sm_60) kernels. Reinstalling
torch==2.5.1/torchvision==0.20.1 (same pin Phase 2 confirmed working) fixes
this.

## 0. Setup: clone repo, reinstall Pascal-compatible torch, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"

!git clone $REPO_URL repo
%cd repo

# Same deliberate exception as Phase 2's training notebook -- see its cell 2
# for the full rationale (preinstalled torch has zero Pascal kernels;
# --extra-index-url not --index-url, or transitive deps like nvidia-cudnn-cu12
# fail to resolve).
!pip install -q torch==2.5.1 torchvision==0.20.1 --extra-index-url https://download.pytorch.org/whl/cu121

!pip install -q -r environment/requirements.txt


## 1. Resolve dataset mount + GPU check

In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("torchvision:", __import__("torchvision").__version__)
print("CUDA available:", torch.cuda.is_available())

gpu_actually_usable = False
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    try:
        torch.zeros(1, device="cuda") + torch.zeros(1, device="cuda")
        gpu_actually_usable = True
        print("GPU FIX CONFIRMED: real CUDA op succeeded")
    except RuntimeError as e:
        print(f"GPU still not usable after torch reinstall: {e}")

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)


## 2. Check for a resume checkpoint from a previous session

In [ ]:
RESUME_CANDIDATE = "checkpoints_resume/mini3drecon_latest.pt"
RESUME_PATH = RESUME_CANDIDATE if os.path.isfile(RESUME_CANDIDATE) else None
print(f"resume checkpoint: {RESUME_PATH or 'none found -- starting from scratch (backbone still pretrained on ImageNet)'}")


## 3. Write a run-specific config (data.root filled in) and run train.py

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

RUN_CONFIG_PATH = "/kaggle/working/run_config.yaml"
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f)
print(f"wrote {RUN_CONFIG_PATH} with data.root = {DATA_ROOT}")


In [ ]:
MAX_STEPS = 20  # smoke test first -- confirm no crash, non-NaN loss components, checkpoint round-trip
                 # before flipping to None for the real 40-epoch run

cmd = [
    "python", "-m", "src.reconstruction.train",
    "--config", "/kaggle/working/run_config.yaml",
    "--output-dir", "/kaggle/working/checkpoints",
]
if MAX_STEPS is not None:
    cmd += ["--max-steps", str(MAX_STEPS)]
if RESUME_PATH is not None:
    cmd += ["--resume", RESUME_PATH]

print("running:", " ".join(cmd))
import subprocess
result = subprocess.run(cmd)
assert result.returncode == 0, f"train.py exited with code {result.returncode}"


## 4. Confirm checkpoints were written

In [ ]:
import os
import torch

ckpt_dir = "/kaggle/working/checkpoints"
files = sorted(os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else []
print("checkpoint files:", files)
assert files, "expected at least one checkpoint file"

epoch_ckpts = sorted((f for f in files if f.startswith("epoch_")), key=lambda f: int(f.split("_")[1].split(".")[0]))
latest = os.path.join(ckpt_dir, epoch_ckpts[-1] if epoch_ckpts else sorted(files)[-1])
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
epoch = ckpt["epoch"]
print(f"loaded {latest}: epoch={epoch}, global_step={ckpt['global_step']}, "
      f"val_depth_absrel={ckpt.get('val_depth_absrel')}, "
      f"val_rot_err_deg={ckpt.get('val_rot_err_deg')}, "
      f"val_trans_err={ckpt.get('val_trans_err')}")
print(f"\n{'TRAINING COMPLETE (epoch 39 reached)' if epoch >= 39 else f'session ended after epoch {epoch} -- resume needed to reach epoch 39'}")


## Done

**If `MAX_STEPS = 20`**: this was the smoke test. Check cell output above for
`GPU FIX CONFIRMED` and non-NaN `depth_loss`/`trans_loss`/`rot_loss` printed
during training -- their raw magnitudes inform whether `pose_rotation_weight:
10.0` in config.yaml needs adjusting before the real run (see PROGRESS.md's
flagged-open note on this). Then set `MAX_STEPS = None` for the real run.

**If `MAX_STEPS = None`**: this was a real training attempt. If it printed
`TRAINING COMPLETE (epoch 39 reached)`, Phase 3 training is done -- record
final val metrics in `PROGRESS.md`. If it printed `session ended after epoch
N`, download the latest `epoch_*.pt`, commit it to
`checkpoints_resume/mini3drecon_latest.pt` in the repo, push, and re-run this
notebook -- it will auto-resume from cell 2's check.